# PAWL-Net: analysis and scene reconstruction

Install `python -m pip install -e ".[notebooks,swin]"` from the repository root.
This notebook replaces the original Colab analysis workflow with shared package imports
and portable paths. Generated plots are new diagnostics, not reproductions of manuscript
figures unless the same source scene and settings are supplied.

See [reported results](../reports/README.md), [research context](../docs/research.md),
and [reproducibility notes](../docs/reproducibility.md).

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from pawl_net.config import load_config
from pawl_net.runtime import resolve_device

config = load_config()
ROOT = config["project_root"]
FIGURES = ROOT / "outputs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
DEVICE = resolve_device(config["training"]["device"])
print("Device:", DEVICE)

## Synthetic cloud morphologies

The implementation chooses one of squares, freeform line strokes, and Perlin noise per
sample. These are the actual generators used by the dataset. No coverage interval is
enforced unless explicitly configured.

In [ ]:
from pawl_net.masks import STRATEGIES, create_cloud_mask

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for index, (axis, strategy) in enumerate(zip(axes, STRATEGIES)):
    mask = create_cloud_mask(128, 128, rng=np.random.default_rng(42 + index), strategy=strategy)
    axis.imshow(mask, cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"{strategy}: {1 - mask.mean():.1%} hidden")
    axis.set_xlabel("Pixel column")
    axis.set_ylabel("Pixel row")
plt.tight_layout()
fig.savefig(FIGURES / "cloud_masks.png", dpi=200, bbox_inches="tight")
plt.show()

## Raw physical channels

The preview uses the first sorted test patch, without selecting a visually favorable
sample. If you use an annual scene instead, choose the month and crop explicitly.
Coordinates reflect the actual array size. Temperatures shown here are raw Kelvin minus
273.15, with source zero-valued NoData hidden; training clipping is not applied to this plot.

In [ ]:
test_files = sorted(config["paths"]["test_patch_dir"].glob("*.npy"))
if test_files:
    sample = np.load(test_files[0], allow_pickle=False)
    layers = [
        (np.where(sample[:, :, 0] > 0, sample[:, :, 0] - 273.15, np.nan), "LST (°C)", "inferno"),
        (sample[:, :, 1], "NDVI", "Greens"),
        (sample[:, :, 2], "NDBI", "Oranges"),
        (sample[:, :, 3], "MNDWI", "Blues"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(9, 8))
    for axis, (layer, title, cmap) in zip(axes.flat, layers):
        plotted = axis.imshow(layer, cmap=cmap)
        axis.set_title(title)
        fig.colorbar(plotted, ax=axis, orientation="horizontal", pad=0.1)
    fig.suptitle(test_files[0].name)
    plt.tight_layout()
    fig.savefig(FIGURES / "physical_channels.pdf", bbox_inches="tight")
    plt.show()
else:
    print("No local test patches. See docs/data.md.")

## Reconstruct a larger scene

Scene inference is opt-in and requires an annual source array, which was not supplied in
the original folder. The example selects March 2024 over Can Tho. Change the source file,
month, and crop deliberately before use. The same synthetic mask is passed to both models.

Real source NoData is combined with the synthetic mask. Real missing pixels have no known
same-date targets; a reconstruction image alone does not validate those pixels.

In [ ]:
from pawl_net.checkpoints import load_model_weights
from pawl_net.experiments import EXPERIMENTS, build_model
from pawl_net.inference import reconstruct_scene

RUN_SCENE_INFERENCE = False
SCENE_PATH = ROOT / "data" / "raw" / "CanTho_2024.npy"
MONTH_INDEX = 2  # zero-based: March
CROP_SIZE = 512

if RUN_SCENE_INFERENCE:
    annual = np.load(SCENE_PATH, mmap_mode="r", allow_pickle=False)
    full_scene = annual[MONTH_INDEX]
    height, width = full_scene.shape[:2]
    if min(height, width) < CROP_SIZE:
        raise ValueError("Selected scene is smaller than the requested crop")
    cy, cx = height // 2, width // 2
    scene = full_scene[
        cy - CROP_SIZE // 2 : cy + CROP_SIZE // 2, cx - CROP_SIZE // 2 : cx + CROP_SIZE // 2
    ].copy()
    known = np.isfinite(scene[:, :, 0]) & (scene[:, :, 0] > 0)
    synthetic = create_cloud_mask(
        *scene.shape[:2], rng=np.random.default_rng(42), strategy="squares"
    )
    observed = known.astype(np.float32) * synthetic
    reconstructions = []
    for name in ("pawl", "swin"):
        experiment = EXPERIMENTS[name]
        model = build_model(experiment, config, pretrained=False)
        load_model_weights(model, config["paths"]["ckpt_dir"] / experiment.best_checkpoint)
        result = reconstruct_scene(model, scene, observed, device=DEVICE)
        reconstructions.append(result)
    truth = np.where(known, scene[:, :, 0] - 273.15, np.nan)
    masked = np.where(observed == 1, truth, np.nan)
    fig, axes = plt.subplots(2, 2, figsize=(10, 9))
    for axis, layer, title in zip(
        axes.flat,
        [truth, masked, *reconstructions],
        ["Reference", "Masked input", "PAWL-Net", "Swin baseline"],
    ):
        im = axis.imshow(layer, cmap="inferno", vmin=19.85, vmax=49.85)
        axis.set_title(title)
    fig.colorbar(im, ax=axes.ravel().tolist(), label="LST (°C)")
    fig.savefig(FIGURES / "scene_comparison.pdf", bbox_inches="tight")
    plt.show()
else:
    print("Set RUN_SCENE_INFERENCE=True after supplying the annual source array.")

## NoData audit

Source zero/NaN values are counted before normalization. The 70% threshold describes
source-data availability and is distinct from the optional synthetic cloud-coverage range.
This cell reads up to 100 patches by default. Set `RUN_FULL_AUDIT=True` for the full folder.

In [ ]:
RUN_FULL_AUDIT = False
audit_files = test_files if RUN_FULL_AUDIT else test_files[:100]
over_70 = entirely_below_clip = 0
for path in audit_files:
    lst = np.load(path, allow_pickle=False)[:, :, 0]
    over_70 += int(np.mean((lst == 0) | ~np.isfinite(lst)) > 0.7)
    entirely_below_clip += int(np.nanmax(lst) < 293)
print(
    {
        "inspected_patches": len(audit_files),
        "over_70_percent_source_nodata": over_70,
        "all_LST_below_lower_clip": entirely_below_clip,
    }
)

## Optional inference timing

Timing includes model execution only, not disk I/O or preprocessing. GPU timing is
synchronized and uses warm-up iterations. These measurements describe your current
hardware and environment; they do not replace manuscript-reported timings.

In [ ]:
def benchmark_model(model, channels=10, iterations=50):
    model = model.to(DEVICE).eval()
    tensor = torch.rand(1, channels, 128, 128, device=DEVICE)
    with torch.inference_mode():
        for _ in range(10):
            model(tensor)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(iterations):
            model(tensor)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    return (time.perf_counter() - start) * 1000 / iterations


RUN_BENCHMARK = False
if RUN_BENCHMARK:
    for name in ("pawl", "swin"):
        experiment = EXPERIMENTS[name]
        model = build_model(experiment, config, pretrained=False)
        load_model_weights(model, config["paths"]["ckpt_dir"] / experiment.best_checkpoint)
        print(name, benchmark_model(model), "ms/patch")
    import joblib

    forest = joblib.load(config["paths"]["ckpt_dir"] / "random_forest_model.joblib")
    pixels = np.random.default_rng(42).random((128 * 128, forest.n_features_in_))
    forest.predict(pixels)  # warm-up
    start = time.perf_counter()
    for _ in range(10):
        forest.predict(pixels)
    print("random-forest", (time.perf_counter() - start) * 100, "ms/patch")